# Data Augmentation a Escala — Dataset BioCatch Vishing (1M filas)



## Objetivo

Escalar el dataset original de 50,000 sesiones a **1,000,000 de sesiones** con una proporción realista de vishing (1-2%), simulando un entorno más cercano a producción donde:

- El volumen de datos es alto (~1M sesiones)
- El desbalance es severo (~1-2% fraude, como en la realidad)
- Las distribuciones y correlaciones de los datos se preservan fielmente

### Estrategia de Augmentación

A diferencia de SMOTE (que interpola entre vecinos), usamos una estrategia de **augmentación paramétrica con perturbación controlada** que:

1. **Estima la distribución de cada feature** por clase (media, std, distribución empírica)
2. **Genera nuevas muestras usando bootstrap con perturbación gaussiana** — toma una muestra real, le añade ruido proporcional a la variabilidad local
3. **Preserva las correlaciones** usando descomposición de Cholesky sobre la matriz de correlación original
4. **Respeta restricciones de dominio** (rangos, tipos enteros, variables binarias, no-negatividad)
5. **Inyecta variabilidad temporal y de cliente** para simular patrones estacionales y diversidad de usuarios

### Output

Un dataset de 1,000,000 de filas con is_vishing al ~1.5%, listo para entrenar modelos en condiciones realistas.


## 1. Setup y Carga de Datos

In [16]:
!pip install -r "requirements.txt"

In [17]:
import boto3
import sagemaker

# Identifica sesión y rol
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
default_bucket = sagemaker_session.default_bucket()

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as scipy_stats
import pyarrow
import fastparquet

plt.rcParams.update({
    'figure.figsize': (14, 6), 'font.size': 11, 'axes.titlesize': 14,
    'axes.labelsize': 12, 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
})

COLORS = {'legit': '#2ecc71', 'vishing': '#e74c3c', 'neutral': '#3498db'}
np.random.seed(42)

print('✅ Librerías cargadas')

✅ Librerías cargadas


In [19]:
s3_input_path = 's3://poc-fraude-vishing/biocatch_sinthetic_data.csv'
df_original = pd.read_csv(s3_input_path, parse_dates=['session_timestamp'])


print(f'Dataset original: {len(df_original):,} filas × {df_original.shape[1]} columnas')
print(f'  Legítimas: {(df_original["is_vishing"]==0).sum():,}')
print(f'  Vishing:   {(df_original["is_vishing"]==1).sum():,} ({df_original["is_vishing"].mean()*100:.1f}%)')

Dataset original: 50,000 filas × 61 columnas
  Legítimas: 47,500
  Vishing:   2,500 (5.0%)


## 2. Clasificación de Features por Tipo

In [20]:
# Columnas de metadata (no se aumentan, se regeneran)
meta_cols = ['session_id', 'customer_id', 'session_timestamp',
             'device_type', 'os_type', 'app_version']

# Label y claim metadata
label_cols = ['is_vishing', 'days_to_claim', 'claim_category']

# Features continuas
continuous_cols = [
    'avg_keyhold_ms', 'avg_interkey_latency_ms', 'typing_speed_cps',
    'keystroke_variability', 'segmented_typing_ratio',
    'avg_touch_pressure', 'avg_touch_size_px', 'swipe_speed_px_s',
    'swipe_directional_variance', 'scroll_speed_avg',
    'device_tilt_angle_mean', 'device_tilt_variability',
    'gyro_rotation_rate_mean', 'accelerometer_jerk_mean',
    'avg_hesitation_duration_s', 'max_hesitation_duration_s',
    'total_dead_time_s', 'dead_time_ratio',
    'screen_transition_time_avg_s', 'data_familiarity_score',
    'session_duration_s', 'call_overlap_duration_s',
    'time_to_transaction_s',
    'errors_per_minute', 'interactions_per_s', 'hesitation_composite',
]

# Features enteras (count-based)
integer_cols = [
    'phone_motion_events', 'hesitation_count', 'dead_time_periods',
    'screens_visited', 'unique_screens_visited', 'unusual_screen_visits',
    'navigation_back_count', 'input_error_count', 'input_correction_count',
    'amount_field_corrections', 'beneficiary_field_corrections',
    'copy_paste_events', 'doodling_events', 'hour_of_day',
    'transaction_amount_cop',
    'biocatch_risk_score', 'biocatch_genuine_score',
]

# Features binarias
binary_cols = [
    'is_atypical_hour', 'phone_call_active',
    'remote_access_tool_detected', 'suspicious_app_detected',
    'transaction_attempted', 'is_new_beneficiary',
    'biocatch_ato_indicator', 'biocatch_social_eng_indicator',
    'biocatch_bot_indicator',
]

# Features con rango [0, 1]
ratio_cols = ['segmented_typing_ratio', 'avg_touch_pressure', 'dead_time_ratio',
              'data_familiarity_score', 'keystroke_variability', 'swipe_directional_variance']

# Todas las features numéricas
all_feature_cols = continuous_cols + integer_cols + binary_cols

print(f'Features continuas: {len(continuous_cols)}')
print(f'Features enteras:   {len(integer_cols)}')
print(f'Features binarias:  {len(binary_cols)}')
print(f'Total features:     {len(all_feature_cols)}')

Features continuas: 26
Features enteras:   17
Features binarias:  9
Total features:     52


## 3. Estrategia de Augmentación: Bootstrap Correlado con Perturbación

La clave para que un millón de filas tengan sentido es preservar la estructura multivariada de los datos, no solo las distribuciones marginales. La estrategia es:

1. **Separar por clase** (legítima vs vishing) — cada clase se aumenta independientemente
2. **Estimar la matriz de correlación** de cada clase sobre las features continuas
3. **Generar datos correlados** usando Cholesky: se genera ruido gaussiano independiente y se transforma con la estructura de correlación original
4. **Mapear a distribuciones marginales reales** usando inverse transform sampling — cada feature se ajusta a la distribución empírica de la clase original
5. **Post-procesar** para respetar tipos, rangos y consistencias lógicas

In [21]:
class CorrelatedAugmenter:
    """
    Genera datos sintéticos que preservan:
    - Distribuciones marginales de cada feature (vía quantile matching)
    - Estructura de correlación entre features (vía Cholesky)
    - Restricciones de dominio (tipos, rangos, no-negatividad)
    """
    
    def __init__(self, df_class, continuous_cols, integer_cols, binary_cols):
        self.continuous_cols = continuous_cols
        self.integer_cols = integer_cols
        self.binary_cols = binary_cols
        self.all_cols = continuous_cols + integer_cols + binary_cols
        
        # Almacenar datos originales para bootstrap y estadísticas
        self.df_class = df_class[self.all_cols].copy()
        self.n_original = len(df_class)
        
        # Calcular estadísticas por feature
        self.means = self.df_class.mean()
        self.stds = self.df_class.std()
        self.mins = self.df_class.min()
        self.maxs = self.df_class.max()
        
        # Probabilidades de las binarias
        self.binary_probs = {col: df_class[col].mean() for col in binary_cols}
        
        # Distribuciones empíricas para integer cols
        self.integer_distributions = {}
        for col in integer_cols:
            vals = df_class[col].values
            unique, counts = np.unique(vals, return_counts=True)
            self.integer_distributions[col] = (unique, counts / counts.sum())
        
        # Matriz de correlación y descomposición de Cholesky para continuas
        cont_data = df_class[continuous_cols].values
        self.corr_matrix = np.corrcoef(cont_data.T)
        # Regularizar para asegurar positive definite
        self.corr_matrix += np.eye(len(continuous_cols)) * 1e-6
        self.cholesky = np.linalg.cholesky(self.corr_matrix)
        
        # Almacenar percentiles para inverse transform
        self.percentiles = {}
        for i, col in enumerate(continuous_cols):
            sorted_vals = np.sort(cont_data[:, i])
            self.percentiles[col] = sorted_vals
    
    def generate(self, n_samples, noise_scale=0.03):
        """
        Genera n_samples filas sintéticas.
        
        noise_scale: controla cuánta variación se añade.
            0.0 = réplicas exactas (bootstrap puro)
            0.03 = perturbación sutil (recomendado)
            0.10 = perturbación alta (más diversidad, menos fidelidad)
        """
        result = pd.DataFrame(index=range(n_samples))
        
        # === CONTINUAS: Cholesky + quantile matching ===
        # Generar ruido gaussiano correlado
        z = np.random.normal(0, 1, (n_samples, len(self.continuous_cols)))
        z_correlated = z @ self.cholesky.T
        
        # Convertir a uniforme vía CDF normal
        u = scipy_stats.norm.cdf(z_correlated)
        
        # Inverse transform: mapear uniforme a distribución empírica
        for i, col in enumerate(self.continuous_cols):
            sorted_vals = self.percentiles[col]
            n_orig = len(sorted_vals)
            # Mapear u[0,1] a índice en sorted_vals
            indices = np.clip((u[:, i] * n_orig).astype(int), 0, n_orig - 1)
            values = sorted_vals[indices]
            
            # Añadir perturbación gaussiana proporcional a la std local
            local_std = self.stds[col] * noise_scale
            values = values + np.random.normal(0, local_std, n_samples)
            
            result[col] = values
        
        # === ENTERAS: muestreo de distribución empírica + perturbación ===
        for col in self.integer_cols:
            unique, probs = self.integer_distributions[col]
            # Muestrear de la distribución empírica
            sampled = np.random.choice(unique, size=n_samples, p=probs)
            # Perturbación: ±1 con probabilidad proporcional a noise_scale
            perturbation = np.random.choice([-1, 0, 0, 0, 1], size=n_samples)
            mask = np.random.random(n_samples) < noise_scale * 5
            sampled = sampled + perturbation * mask
            result[col] = sampled
        
        # === BINARIAS: muestreo Bernoulli con probabilidades originales ===
        for col in self.binary_cols:
            prob = self.binary_probs[col]
            result[col] = (np.random.random(n_samples) < prob).astype(int)
        
        # === POST-PROCESAMIENTO ===
        self._enforce_constraints(result)
        
        return result
    
    def _enforce_constraints(self, df):
        """Aplica restricciones de dominio."""
        # Ratios [0, 1]
        for col in ['segmented_typing_ratio', 'avg_touch_pressure', 'dead_time_ratio',
                     'data_familiarity_score', 'keystroke_variability', 'swipe_directional_variance']:
            if col in df.columns:
                df[col] = df[col].clip(0, 1)
        
        # Enteros no negativos
        for col in self.integer_cols:
            if col in df.columns:
                df[col] = df[col].round().clip(lower=0).astype(int)
        
        # Binarias
        for col in self.binary_cols:
            if col in df.columns:
                df[col] = df[col].round().clip(0, 1).astype(int)
        
        # Scores BioCatch [0, 1000]
        for col in ['biocatch_risk_score', 'biocatch_genuine_score']:
            if col in df.columns:
                df[col] = df[col].clip(0, 1000)
        
        # Hora del día [0, 23]
        if 'hour_of_day' in df.columns:
            df['hour_of_day'] = df['hour_of_day'].clip(0, 23)
        
        # Ángulo de inclinación [0, 90]
        if 'device_tilt_angle_mean' in df.columns:
            df['device_tilt_angle_mean'] = df['device_tilt_angle_mean'].clip(0, 90)
        
        # Non-negative para todas las continuas que lo eran originalmente
        for col in self.continuous_cols:
            if col in df.columns and self.mins[col] >= 0:
                df[col] = df[col].clip(lower=0)
        
        # Consistencias lógicas
        if 'unique_screens_visited' in df.columns and 'screens_visited' in df.columns:
            df['unique_screens_visited'] = np.minimum(
                df['unique_screens_visited'], df['screens_visited'])
        
        if 'call_overlap_duration_s' in df.columns and 'phone_call_active' in df.columns:
            df.loc[df['phone_call_active'] == 0, 'call_overlap_duration_s'] = 0
        
        if 'time_to_transaction_s' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'time_to_transaction_s'] = 0
        
        if 'transaction_amount_cop' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'transaction_amount_cop'] = 0
        
        if 'is_new_beneficiary' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'is_new_beneficiary'] = 0


print('✅ CorrelatedAugmenter definido')

✅ CorrelatedAugmenter definido


## 4. Generación del Dataset Aumentado (1,000,000 sesiones)

Configuración objetivo:
- **Total**: 1,000,000 sesiones
- **is_vishing=1**: ~1.5% (~15,000 sesiones)
- **is_vishing=0**: ~98.5% (~985,000 sesiones)
- Todas las 50,000 sesiones originales se incluyen intactas

In [22]:
# Configuración
N_TOTAL = 1_000_000
VISHING_PCT = 0.015  # 1.5%

n_vishing_target = int(N_TOTAL * VISHING_PCT)
n_legit_target = N_TOTAL - n_vishing_target

# Datos originales por clase
df_orig_legit = df_original[df_original['is_vishing'] == 0]
df_orig_vishing = df_original[df_original['is_vishing'] == 1]

n_legit_to_generate = n_legit_target - len(df_orig_legit)
n_vishing_to_generate = n_vishing_target - len(df_orig_vishing)

print('PLAN DE GENERACIÓN')
print('='*60)
print(f'Target total:           {N_TOTAL:>12,}')
print(f'Target vishing ({VISHING_PCT*100}%): {n_vishing_target:>12,}')
print(f'Target legítimas:       {n_legit_target:>12,}')
print(f'\nOriginales legítimas:   {len(df_orig_legit):>12,}')
print(f'Originales vishing:     {len(df_orig_vishing):>12,}')
print(f'\nA GENERAR legítimas:    {n_legit_to_generate:>12,}')
print(f'A GENERAR vishing:      {n_vishing_to_generate:>12,}')
print(f'\nMultiplicador legítimas: {n_legit_target/len(df_orig_legit):.1f}x')
print(f'Multiplicador vishing:   {n_vishing_target/len(df_orig_vishing):.1f}x')

PLAN DE GENERACIÓN
Target total:              1,000,000
Target vishing (1.5%):       15,000
Target legítimas:            985,000

Originales legítimas:         47,500
Originales vishing:            2,500

A GENERAR legítimas:         937,500
A GENERAR vishing:            12,500

Multiplicador legítimas: 20.7x
Multiplicador vishing:   6.0x


In [23]:
# Crear augmenters por clase
print('Construyendo augmenter para clase LEGÍTIMA...')
t0 = time.time()
aug_legit = CorrelatedAugmenter(df_orig_legit, continuous_cols, integer_cols, binary_cols)
print(f'  ✅ Listo ({time.time()-t0:.1f}s)')

print('\nConstruyendo augmenter para clase VISHING...')
t0 = time.time()
aug_vishing = CorrelatedAugmenter(df_orig_vishing, continuous_cols, integer_cols, binary_cols)
print(f'  ✅ Listo ({time.time()-t0:.1f}s)')

Construyendo augmenter para clase LEGÍTIMA...
  ✅ Listo (0.1s)

Construyendo augmenter para clase VISHING...
  ✅ Listo (0.0s)


In [24]:
# Generar datos sintéticos
print('Generando datos sintéticos de clase LEGÍTIMA...')
t0 = time.time()
df_synth_legit = aug_legit.generate(n_legit_to_generate, noise_scale=0.03)
print(f'  ✅ {len(df_synth_legit):,} filas en {time.time()-t0:.1f}s')

print('\nGenerando datos sintéticos de clase VISHING...')
t0 = time.time()
df_synth_vishing = aug_vishing.generate(n_vishing_to_generate, noise_scale=0.03)
print(f'  ✅ {len(df_synth_vishing):,} filas en {time.time()-t0:.1f}s')

Generando datos sintéticos de clase LEGÍTIMA...
  ✅ 937,500 filas en 4.5s

Generando datos sintéticos de clase VISHING...
  ✅ 12,500 filas en 0.1s


In [25]:
# Ensamblar dataset completo
print('Ensamblando dataset final...')
t0 = time.time()

# Originales
df_orig_legit_features = df_orig_legit[all_feature_cols].copy()
df_orig_legit_features['is_vishing'] = 0
df_orig_legit_features['is_synthetic'] = 0

df_orig_vishing_features = df_orig_vishing[all_feature_cols].copy()
df_orig_vishing_features['is_vishing'] = 1
df_orig_vishing_features['is_synthetic'] = 0

# Sintéticas
df_synth_legit['is_vishing'] = 0
df_synth_legit['is_synthetic'] = 1

df_synth_vishing['is_vishing'] = 1
df_synth_vishing['is_synthetic'] = 1

# Concatenar
df_augmented = pd.concat([
    df_orig_legit_features,
    df_orig_vishing_features,
    df_synth_legit,
    df_synth_vishing,
], ignore_index=True)

# Shuffle
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

# Generar IDs
df_augmented.insert(0, 'session_id', [f'SES-{i:07d}' for i in range(1, len(df_augmented)+1)])
df_augmented.insert(1, 'customer_id', [f'CUS-{np.random.randint(10000, 99999)}' for _ in range(len(df_augmented))])

# Generar timestamps distribuidos en 12 meses
from datetime import datetime, timedelta
base_date = datetime(2024, 6, 1)
timestamps = [base_date + timedelta(
    days=np.random.randint(0, 365),
    hours=int(df_augmented.iloc[i]['hour_of_day']),
    minutes=np.random.randint(0, 60),
    seconds=np.random.randint(0, 60)
) for i in range(len(df_augmented))]
df_augmented.insert(2, 'session_timestamp', timestamps)

# Metadata de dispositivo
df_augmented.insert(3, 'device_type', np.where(np.random.rand(len(df_augmented)) < 0.85, 'mobile', 'web'))
df_augmented.insert(4, 'os_type', np.where(
    df_augmented['device_type'] == 'mobile',
    np.where(np.random.rand(len(df_augmented)) < 0.6, 'Android', 'iOS'),
    np.where(np.random.rand(len(df_augmented)) < 0.7, 'Windows', 'macOS')))
df_augmented.insert(5, 'app_version', np.random.choice(
    ['v12.1', 'v12.2', 'v12.3', 'v13.0', 'v13.1'], len(df_augmented)))

# Claim metadata
df_augmented['days_to_claim'] = df_augmented['is_vishing'].apply(
    lambda v: int(np.clip(np.random.exponential(3), 0, 30)) if v == 1 else -1)
df_augmented['claim_category'] = df_augmented['is_vishing'].apply(
    lambda v: np.random.choice(['vishing', 'ingenieria_social_telefonica', 'fraude_telefono'],
                               p=[0.6, 0.3, 0.1]) if v == 1 else 'none')

elapsed = time.time() - t0
print(f'\n✅ Dataset ensamblado en {elapsed:.1f}s')
print(f'   Filas: {len(df_augmented):,}')
print(f'   Columnas: {df_augmented.shape[1]}')
print(f'   Vishing: {(df_augmented["is_vishing"]==1).sum():,} ({df_augmented["is_vishing"].mean()*100:.2f}%)')
print(f'   Sintéticas: {(df_augmented["is_synthetic"]==1).sum():,} ({df_augmented["is_synthetic"].mean()*100:.1f}%)')

Ensamblando dataset final...

✅ Dataset ensamblado en 76.3s
   Filas: 1,000,000
   Columnas: 62
   Vishing: 15,000 (1.50%)
   Sintéticas: 950,000 (95.0%)


In [31]:
# Guardar dataset aumentado
#os.makedirs('data/augmented', exist_ok=True)
output_path = 's3://poc-fraude-vishing/data/augmented/dataset_1M_vishing_.parquet'

print(f'Guardando dataset en {output_path}...')
t0 = time.time()
df_augmented.to_parquet(output_path, engine='pyarrow', index=False)

Guardando dataset en s3://poc-fraude-vishing/data/augmented/dataset_1M_vishing_.parquet...
